# Fine-Tune the Door Detector on Real Plans

The durable fix for the review rate. Your `door_yolo11.pt` was trained on the Roboflow `doorplan` style; real firm drawings differ, so it *finds* doors but isn't *confident* (baseline: ~38% of detections flagged for review, and 383/544 on the Sanibel set).

This notebook continues training from your existing weights on **corrected labels from real plans** — transfer learning, so it converges fast on a small dataset. Target: review rate **< 20%**.

**Prerequisites:**
1. Ran `backend/scripts/prelabel.py` locally to generate pre-labeled pages.
2. Uploaded them to Roboflow and **corrected** the boxes (see `docs/FINE_TUNING.md` §3).
3. Exported the corrected dataset as **YOLOv11** format.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 0. Setup

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
%pip -q install ultralytics roboflow
from ultralytics import YOLO
from pathlib import Path
import shutil, glob

WEIGHTS = Path('/content/weights'); WEIGHTS.mkdir(exist_ok=True)

## 1. Upload your current weights

Fine-tuning **must start from `door_yolo11.pt`**, not `yolo11n.pt` — starting from the base model would throw away everything it already learned from the `doorplan` dataset.

In [ ]:
from google.colab import files
print('Upload door_yolo11.pt (from backend/models/):')
up = files.upload()
BASE_WEIGHTS = '/content/' + list(up)[0]
print('base weights ->', BASE_WEIGHTS)

## 2. Download your corrected dataset from Roboflow

On your Roboflow project: **Versions → Create New Version** (train/valid/test split ≈ 70/20/10), then **Export → YOLOv11 → show download code** and paste the snippet below.

Augmentation advice for line-art blueprints: keep it *modest* — slight rotation (±5°), brightness, mild blur. **Avoid heavy color/hue jitter**; drawings are black-on-white line art, so color augmentation is unrealistic and hurts.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = ''   # <-- Settings -> API -> Private API Key
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# ---- PASTE Roboflow's exact download lines here ----
project = rf.workspace('YOUR_WORKSPACE').project('fire-door-real-plans')
version = project.version(1)
# ----------------------------------------------------

ds = version.download('yolov11')
DATA_YAML = ds.location + '/data.yaml'
print(open(DATA_YAML).read())   # sanity-check: expect a single 'door' class

## 3. Measure the BEFORE baseline

Run the *current* model against the new real-plan validation set first. Without this number you can't honestly claim the fine-tune improved anything.

In [ ]:
before = YOLO(BASE_WEIGHTS).val(data=DATA_YAML, imgsz=1024)
print(f'BEFORE  mAP50: {before.box.map50:.4f} | mAP50-95: {before.box.map:.4f} | recall: {before.box.mr:.4f}')

## 4. Fine-tune

Settings chosen to **refine, not overwrite**:
- `lr0=0.001` — 10× lower than the from-scratch default, so updates are small and careful.
- `freeze=10` — freeze early layers; they already detect generic edges/arcs well. Mainly retrains the head on your drawing styles, which also prevents overfitting on a small dataset.
- `patience=15` — early-stop when validation plateaus.
- `mosaic=0.5` — lighter mosaic augmentation; heavy mosaic is unrealistic for line art.

In [ ]:
model = YOLO(BASE_WEIGHTS)          # start from YOUR weights, NOT yolo11n.pt
model.train(
    data=DATA_YAML,
    epochs=40,
    imgsz=1024,
    batch=8,
    lr0=0.001,
    freeze=10,
    patience=15,
    mosaic=0.5,
    project='runs', name='door_finetune',
)

## 5. Measure the AFTER — did it actually improve?

**Watch recall especially.** For a fire-safety product a *missed* door is the worst error, so recall must not drop even if precision rises.

In [ ]:
after = model.val(data=DATA_YAML, imgsz=1024)
print(f'BEFORE  mAP50: {before.box.map50:.4f} | mAP50-95: {before.box.map:.4f} | recall: {before.box.mr:.4f}')
print(f'AFTER   mAP50: {after.box.map50:.4f} | mAP50-95: {after.box.map:.4f} | recall: {after.box.mr:.4f}')
print()
print(f'mAP50 change:  {after.box.map50 - before.box.map50:+.4f}')
print(f'recall change: {after.box.mr - before.box.mr:+.4f}   <-- must not go meaningfully negative')

## 6. Visual spot-check

Numbers can improve while the model does something silly. Look at actual predictions before shipping the weights.

In [ ]:
samples = glob.glob(ds.location + '/valid/images/*')[:3]
results = model.predict(samples, imgsz=1024, conf=0.25, save=True)

from IPython.display import Image as IPyImage, display
for p in glob.glob(str(Path(results[0].save_dir) / '*'))[:3]:
    display(IPyImage(filename=p, width=900))

## 7. Download the new weights

Replace `backend/models/door_yolo11.pt` with this file, then re-run detection locally and compare the **review rate** (`needs_review / total_doors`) against the 38% baseline. Log the result in `docs/PROGRESS.md`.

**Keep the old weights** until the new ones are verified on real documents — a regression is easier to undo than to diagnose.

In [ ]:
best = model.trainer.best
shutil.copy(best, WEIGHTS / 'door_yolo11_finetuned.pt')
print('best weights:', best)
files.download(str(WEIGHTS / 'door_yolo11_finetuned.pt'))

## 8. Iterate (active learning)

The efficient way to keep improving with minimal labeling:
1. Run the new model on **fresh** real plans.
2. Find pages where it's still unsure (lots of amber) or wrong.
3. Pre-label + correct **those** pages — hard cases teach the most.
4. Add them to the Roboflow project, create a new version, re-run this notebook.

Each loop targets the model's actual weak spots instead of re-teaching what it already knows.